# Applied Machine Learning, Notebook 8
## Detecting fraud without labels: PCA as an anomaly detector

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### A problem where the label arrives late

In notebooks 5, 6 and 7 we always had the correct answer to learn from: we knew which customers had
stopped paying, and the model learned to recognize them. That is called **supervised** learning. In card
fraud that comfort barely exists:

- **The label arrives late.** A transaction is marked as fraud when the customer files a claim, days or
  weeks later. The model has to decide in milliseconds, long before anyone knows.
- **The label is scarce.** In today's dataset, fewer than 2 out of every 1,000 transactions are fraud. With so
  few examples, a supervised model has very little to learn from.
- **Fraud changes.** Fraudsters experiment, see what gives them away and change their method. A model that
  only recognizes the fraud it has already seen is blind to new fraud.

The alternative is to turn the question around. Instead of learning what fraud looks like, we learn **what a
normal transaction looks like**, and we flag as suspicious any transaction that does not fit that pattern. This is
called **anomaly detection**, and it is **unsupervised** learning: it does not use the label to learn.

Today's tool is **principal component analysis** (PCA).
PCA finds the few directions in which normal data move. With those few directions we can
**reconstruct** a normal transaction almost perfectly; a transaction that cannot be reconstructed
well is doing something normal transactions do not do. The size of that reconstruction error is our
**suspicion score**.

### What you will learn today

1. The idea of PCA with two variables, in a picture: project, reconstruct and measure the error.
2. The minimal math: explained variance, **loadings**, **scores** and reconstruction.
3. The scikit-learn commands, one by one: `fit`, `explained_variance_ratio_`, `components_`,
   `transform`, `inverse_transform`, and how to go back to the original units.
4. How to turn the reconstruction error into a **fraud detector**, how to set the threshold without
   labels and how to evaluate it when the labels arrive.
5. Why it works, why it can stop working (the **contamination** of what we call normal) and how to
   fix it.
6. The honest comparison with the **supervised** models from previous notebooks.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os, urllib.request, warnings

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

## 1. The data: 284,807 card transactions

We use the public **Credit Card Fraud Detection** dataset (Dal Pozzolo et al., 2015; Université Libre de
Bruxelles and Worldline): all card transactions of a European issuer over two days in
September 2013. The column `Class` equals 1 if the transaction turned out to be fraud.

There is one detail you need to know from the start. For confidentiality, the issuer did not release the
original variables (merchant, country, card type, etc.): it released 28 columns `V1` to `V28` that are
**principal components** of those variables, that is, the result of applying to them precisely the PCA we are going
to learn today. Only two columns come in their original scale: `Time` (seconds since the first
transaction in the dataset) and `Amount` (amount, in euros). This means we cannot give `V1` to `V28` an economic
interpretation, an honest limitation of this dataset, and later on it will bring us a
surprise.

The file is about 150 MB. The cell downloads it only once; if you have the course copy, paste its link
into `URL_FRAUDE`.

In [2]:
URL_FRAUDE = ""    # <-- paste the link to the course CSV (creditcard.csv) here if you have it

def normalizar_link(url):
    '''Turns a Dropbox or Google Drive link into a direct download link.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if not os.path.exists("creditcard.csv"):
    fuente = normalizar_link(URL_FRAUDE) if URL_FRAUDE else \
        "https://huggingface.co/datasets/David-Egea/Creditcard-fraud-detection/resolve/main/creditcard.csv"
    print("Downloading creditcard.csv (150 MB, a couple of minutes)...")
    urllib.request.urlretrieve(fuente, "creditcard.csv")
datos = pd.read_csv("creditcard.csv")

print("Transactions:", f"{len(datos):,}", "  Frauds:", int(datos["Class"].sum()))
print("Fraud rate:", round(100 * datos["Class"].mean(), 3), "%  (1 in every", round(1 / datos["Class"].mean()), ")")
print("Hours covered:", round(datos["Time"].max() / 3600, 1))
datos.groupby("Class")["Amount"].describe()[["count", "mean", "50%", "max"]].round(2)

Transactions: 284,807   Frauds: 492
Fraud rate: 0.173 %  (1 in every 579 )
Hours covered: 48.0


,count,mean,50%,max
Class,,,,
0,284315.0,88.29,22.00,25691.16
1,492.0,122.21,9.25,2125.87


Fraud is **0.17%** of transactions: 492 out of 284,807. A "model" that says nothing is fraud
is right 99.83% of the time, so accuracy is useless here. We will measure as in
notebook 5, looking at the class that matters: **recall** (of the frauds, how many we catch), **precision** (of
the alerts, how many were fraud) and their summary over all thresholds, the **PR-AUC** (area under the
precision-recall curve), which with such imbalanced classes is more informative than the AUC of the ROC curve. And one
operational metric: if the fraud team can review **100 alerts**, how many frauds are among the 100
most suspicious transactions? That is **precision@100**.

## 2. The idea of PCA with two variables

Before the 29 columns, a picture with two. Imagine a rewards program: with each purchase the card
earns points, roughly proportional to the amount. We simulate 500 normal transactions with two variables
already standardized (mean 0, standard deviation 1): `monto` (amount) and `puntos` (points), with correlation 0.9. We add two
special transactions:

- **A**: a very large purchase with the points it should earn. It is extreme, but it **fits the pattern**.
- **B**: a purchase with moderate amount and points, but with many more points than the amount should earn.
  Neither of its two values is unusual on its own; what is unusual is **the combination**. It is what you would see if
  someone were manipulating the points system.

In [3]:
rng = np.random.default_rng(7)
n = 500
monto = rng.normal(0, 1, n)
puntos = 0.9 * monto + np.sqrt(1 - 0.9**2) * rng.normal(0, 1, n)
juguete = pd.DataFrame({"monto": monto, "puntos": puntos, "tipo": "normal"})
especiales = pd.DataFrame({"monto": [2.6, -0.4], "puntos": [2.5, 1.3], "tipo": ["A: extreme but fits", "B: does not fit"]})
juguete = pd.concat([juguete, especiales], ignore_index=True)
X2 = juguete[["monto", "puntos"]].values

pca2 = PCA(n_components=2).fit(X2[:n])        # fitted only with the 500 normal ones
print("Variance explained by each component:", pca2.explained_variance_ratio_.round(3))
print("Component 1 (loadings):", pca2.components_[0].round(3))
print("Component 2 (loadings):", pca2.components_[1].round(3))

Variance explained by each component: [0.95 0.05]
Component 1 (loadings): [0.706 0.708]
Component 2 (loadings): [ 0.708 -0.706]


PCA found two directions. The **first component** has similar loadings on amount and points (both
with the same sign): it is the direction "bigger purchase, more points", and on its own it explains 95% of the
variation. The **second** is perpendicular: "more points than the amount should earn", and it explains only
5%. In PCA language, normal transactions live almost entirely on a line.

Now the trick. If we keep only the first component, each transaction is summarized by **one number**
(its position along the line, the *score*), and from that number we can **reconstruct** an approximate
version of the original transaction. The distance between the original and the reconstruction is the **reconstruction
error**.

In [4]:
pca1 = PCA(n_components=1).fit(X2[:n])
puntaje = pca1.transform(X2)                  # one number per transaction
reconstruida = pca1.inverse_transform(puntaje) # back to (monto, puntos)
juguete["error"] = ((X2 - reconstruida) ** 2).sum(axis=1)

fig = go.Figure()
for tipo, color in [("normal", "#94a3b8"), ("A: extreme but fits", "#2563eb"), ("B: does not fit", "#dc2626")]:
    m = juguete["tipo"] == tipo
    fig.add_scatter(x=juguete.loc[m, "monto"], y=juguete.loc[m, "puntos"], mode="markers", name=tipo,
                    marker=dict(color=color, size=7 if tipo == "normal" else 14, opacity=0.6 if tipo == "normal" else 1))
for i in [n, n + 1]:
    fig.add_scatter(x=[X2[i, 0], reconstruida[i, 0]], y=[X2[i, 1], reconstruida[i, 1]], mode="lines",
                    line=dict(color="black", dash="dot"), showlegend=False)
eje = np.linspace(-3.2, 3.2, 2)[:, None] * pca1.components_[0] + pca1.mean_
fig.add_scatter(x=eje[:, 0], y=eje[:, 1], mode="lines", name="component 1", line=dict(color="#16a34a", width=3))
fig.update_layout(title="PCA with two variables: the line of normal transactions and the reconstruction error (dotted)",
                  xaxis_title="amount (standardized)", yaxis_title="points (standardized)", height=480,
                  yaxis_scaleanchor="x")
fig.show()
print(juguete.loc[n:, ["tipo", "monto", "puntos", "error"]].round(3).to_string(index=False))
print("Reconstruction error of the normal transactions: 99th percentile =", round(juguete.loc[:n - 1, "error"].quantile(0.99), 3))

               tipo  monto  puntos  error
A: extreme but fits    2.6     2.5  0.006
    B: does not fit   -0.4     1.3  1.432
Reconstruction error of the normal transactions: 99th percentile = 0.516


This is the whole idea of the notebook in one chart. **A** is the most extreme transaction of all, but its
error is almost zero: it falls on the line, and the line reconstructs it well. **B** has moderate values in both
variables, but its error (1.43) is almost three times the 99th percentile of the normal ones (0.52): there is no
point on the line that resembles it. A detector based on reconstruction error does not look for **large** transactions, it looks for
transactions that **break the relationships** that normal ones follow. That is exactly what we want for
fraud, where often each piece of data on its own looks innocent.

## 3. The minimal math

With $p$ variables, PCA does the same as in the picture but in $p$ dimensions. The steps, by name:

**Standardize.** From each variable $j$ we subtract its mean and divide by its standard deviation, both
computed in training: $x_j \leftarrow (x_j - \bar{x}_j) / s_j$. That way no variable dominates just because it is on
a larger scale.

**Eigenvalue and eigenvector decomposition** (Pearson, 1901; Hotelling, 1933). With the covariance
matrix $S$ of the standardized data we look for the directions $w_j$ that satisfy

$$ S\, w_j = \lambda_j\, w_j, \qquad j = 1, \dots, p $$

- $S$: covariance matrix, $p \times p$ (with standardized data it is the correlation matrix).
- $w_j$: **eigenvector** $j$, a vector of length 1 with $p$ numbers. Its elements are the **loadings**:
  how much each original variable weighs in that direction. In scikit-learn it is row $j$ of `components_`.
- $\lambda_j$: **eigenvalue** $j$, the variance of the data along $w_j$. They are sorted from largest to
  smallest, $\lambda_1 \ge \lambda_2 \ge \dots$. In scikit-learn it is `explained_variance_`.
- The **explained variance** of component $j$ is $\lambda_j / \sum_{i=1}^{p} \lambda_i$
  (`explained_variance_ratio_`). With standardized data $\sum_i \lambda_i = p$.

**Scores and reconstruction.** If we keep the first $k$ components (the columns of $W_k$, which is
$p \times k$), a transaction $x$ (already standardized, centered at $\mu$) is summarized by $k$ numbers and reconstructed like this:

$$ z = W_k^{\top} (x - \mu), \qquad \hat{x} = \mu + W_k\, z $$

- $z$: the vector of $k$ **scores**, the position of the transaction along each direction. It is `transform`.
- $\hat{x}$: the **reconstruction**, back in the $p$ variables. It is `inverse_transform`.
- $\mu$: the vector of training means (`mean_`; with standardized data it is almost zero).

**Reconstruction error.** Our suspicion score is the squared distance between the original and its
reconstruction, which in the process control literature is called the $Q$ statistic or SPE, *squared
prediction error* (Jackson and Mudholkar, 1979):

$$ e(x) = \lVert x - \hat{x} \rVert^2 = \sum_{j=1}^{p} (x_j - \hat{x}_j)^2 = \sum_{j=k+1}^{p} z_j^2 $$

The last equality says something important: the error is the sum of the squared scores on the components
we **discard**. A transaction has a large error if it moves a lot in directions where normal transactions
barely move. In the picture, $p = 2$, $k = 1$ and the error is the squared score on component 2.

We verify it with the numbers from the picture: the formula by hand has to give the same as scikit-learn.

In [5]:
W_k = pca1.components_.T                      # p x k  (here 2 x 1)
mu = pca1.mean_
z = (X2 - mu) @ W_k                           # scores by hand
x_hat = mu + z @ W_k.T                        # reconstruction by hand
e_mano = ((X2 - x_hat) ** 2).sum(axis=1)
z_todos = pca2.transform(X2)                  # both components
print("Scores by hand = transform?                ", np.allclose(z, puntaje))
print("Reconstruction by hand = inverse_transform?", np.allclose(x_hat, reconstruida))
print("Error = discarded score squared?           ", np.allclose(e_mano, z_todos[:, 1] ** 2))

Scores by hand = transform?                 True
Reconstruction by hand = inverse_transform? True
Error = discarded score squared?            True


## 4. Preparing the real data

Three preparation decisions, each with its reason:

1. **Which columns.** We use `V1` to `V28` and `Amount`, 29 in total. We leave out `Time`: it is not a
   characteristic of the transaction but a clock that only grows (seconds since the start of the dataset), and a
   model that learns on one range of hours and is applied to another would be extrapolating. One of the exercises
   shows what happens if it is included.
2. **Training and testing.** We split 67% for learning and 33% for measuring, at random but keeping the
   fraud proportion in both (stratified), with the usual seed. PCA is fitted **only**
   on training and **without looking at the label**; the testing label is used only to evaluate at the end.
3. **Standardize.** `Amount` has a standard deviation of 250 euros and the `V` columns about 1. Without standardizing,
   PCA would say that the first component is the amount and nothing else, just because of the scale. Let us check.

In [6]:
COLS = [f"V{i}" for i in range(1, 29)] + ["Amount"]
tr, te = train_test_split(datos, test_size=0.33, random_state=2018, stratify=datos["Class"])
y_tr, y_te = tr["Class"].values, te["Class"].values
print("training:", len(tr), "transactions,", y_tr.sum(), "frauds  |  testing:", len(te), "transactions,", y_te.sum(), "frauds")

sin_escalar = PCA().fit(tr[COLS])
print("\nWithout standardizing, variance explained by component 1:", round(sin_escalar.explained_variance_ratio_[0], 4))
print("Loading of Amount on component 1:", round(sin_escalar.components_[0][-1], 4))

escalador = StandardScaler().fit(tr[COLS])     # training mean and standard deviation
X_tr = escalador.transform(tr[COLS])
X_te = escalador.transform(te[COLS])

training: 190820 transactions, 330 frauds  |  testing: 93987 transactions, 162 frauds

Without standardizing, variance explained by component 1: 0.9995
Loading of Amount on component 1: 1.0


Without standardizing, the first component "explains" 99.9% of the variance and is pure `Amount` (loading
practically 1): it is not a finding, it is an artifact of the units. From here on we work with `X_tr` and
`X_te` standardized with the training mean and standard deviation.

## 5. PCA in scikit-learn, command by command

### 5.1 Fit and look at the explained variance

`PCA()` with no arguments computes all the components (here 29). `fit` estimates means, loadings and
eigenvalues on training.

In [7]:
pca_full = PCA().fit(X_tr)
var_exp = pca_full.explained_variance_ratio_
tabla_var = pd.DataFrame({"component": np.arange(1, 30), "eigenvalue": pca_full.explained_variance_,
                          "explained variance": var_exp, "cumulative": var_exp.cumsum()})
print(tabla_var.round(4).iloc[[0, 1, 2, 4, 9, 19, 24, 25, 26, 27, 28]].to_string(index=False))

fig = go.Figure()
fig.add_bar(x=tabla_var["component"], y=tabla_var["explained variance"], name="per component")
fig.add_scatter(x=tabla_var["component"], y=tabla_var["cumulative"], name="cumulative", mode="lines+markers", yaxis="y2")
fig.add_hline(y=0.95, line_dash="dot", annotation_text="95%", yref="y2")
fig.update_layout(title="Variance explained by each component (29 standardized columns)", xaxis_title="component",
                  yaxis_title="per component", yaxis2=dict(overlaying="y", side="right", range=[0, 1.02], title="cumulative"), height=400)
fig.show()

 component  eigenvalue  explained variance  cumulative
         1      1.9344              0.0667      0.0667
         2      1.0549              0.0364      0.1031
         3      1.0442              0.0360      0.1391
         5      1.0211              0.0352      0.2098
        10      1.0064              0.0347      0.3840
        20      0.9947              0.0343      0.7289
        25      0.9805              0.0338      0.8994
        26      0.9745              0.0336      0.9330
        27      0.9682              0.0334      0.9664
        28      0.9334              0.0322      0.9986
        29      0.0414              0.0014      1.0000


This chart is very different from the one in textbooks, where two or three components explain almost everything.
Here each component explains between 3% and 4%, except the first (7%) and the last (0.14%). The reason is the one
we announced: `V1` to `V28` **already are** principal components, they are uncorrelated with each other, and once
standardized each one contributes the same. PCA only finds structure where there is some: in the relationship of
`Amount` with some of the `V` columns. To reach 95% of the variance we need **27 of 29 components**.

What looks like a failure of dimensionality "reduction" is actually fine for an anomaly detector.
We are not interested in summarizing the data in a few columns; we are interested in finding the **few directions where
normal transactions barely move**. Those are the last ones: 28 and, above all, 29.

### 5.2 The loadings: what each component is

`components_` is a matrix of 29 components by 29 variables: row $j$ holds the loadings $w_j$. We
read it as a table for the first and the last two.

In [8]:
cargas = pd.DataFrame(pca_full.components_, columns=COLS, index=[f"CP{j}" for j in range(1, 30)])

def top_cargas(fila, n=6):
    s = cargas.loc[fila]
    return s.reindex(s.abs().sort_values(ascending=False).index)[:n].round(2)

for fila in ["CP1", "CP28", "CP29"]:
    print(fila, "| explained variance:", round(var_exp[int(fila[2:]) - 1], 4))
    print("   ", top_cargas(fila).to_dict())

fig = px.imshow(cargas.iloc[[0, 1, 26, 27, 28]], color_continuous_scale="RdBu_r", zmin=-0.75, zmax=0.75, aspect="auto",
                title="Loadings of some components (rows) on the 29 variables (columns)")
fig.update_layout(height=330)
fig.show()

CP1 | explained variance: 0.0667
    {'Amount': 0.71, 'V2': -0.39, 'V7': 0.31, 'V5': -0.3, 'V20': 0.22, 'V1': -0.17}
CP28 | explained variance: 0.0322
    {'V17': 0.41, 'V12': 0.36, 'V7': 0.35, 'V14': 0.33, 'V10': 0.31, 'V16': 0.29}
CP29 | explained variance: 0.0014
    {'Amount': 0.7, 'V2': 0.39, 'V7': -0.3, 'V5': 0.29, 'V20': -0.25, 'V1': 0.17}


- **CP1** (the one that explains the most variance): loading 0.71 on `Amount` and loadings of opposite sign on `V2`, `V5` and
  of the same sign on `V7`, `V20`. It is the "amount" direction, together with the `V` columns that accompany the amount in a normal
  purchase.
- **CP29** (the one that explains the least variance, 0.14%): **the same variables with the signs reversed**. It is the
  direction "an amount that does not match its `V` columns". In a normal purchase `Amount` can be predicted almost
  perfectly from `V2`, `V5`, `V7` and `V20`, and this component measures how far a transaction departs from that rule.
  It is the exact analog of component 2 in the points picture.
- **CP28**: positive loadings on `V17`, `V12`, `V14`, `V10`, `V16` and `V7`. Keep this list in mind; in section 7
  we will see why it is the one that matters most.

### 5.3 Choosing how many components: `n_components`

There are three ways to tell scikit-learn how many components to keep: an integer (`PCA(n_components=27)`), a
share of variance (`PCA(n_components=0.95)`, which picks the smallest $k$ that reaches 95%) or nothing (all of them).
We use the 95% rule, which needs no labels.

In [9]:
pca = PCA(n_components=0.95).fit(X_tr)
k = pca.n_components_
print("Components kept with the 95% rule:", k, " | explained variance:", round(pca.explained_variance_ratio_.sum(), 4))
print("Shape of components_:", pca.components_.shape, "  (k components x 29 variables)")

Components kept with the 95% rule: 27  | explained variance: 0.9664
Shape of components_: (27, 29)   (k components x 29 variables)


### 5.4 There and back: `transform`, `inverse_transform` and the original units

`transform` takes each transaction from 29 variables to its $k$ scores. `inverse_transform` makes the way
back: from the scores it reconstructs the 29 **standardized** variables. And to return to the original
units (euros in `Amount`) we undo the standardization with `escalador.inverse_transform`. Let us look at one
normal transaction and one fraud from testing, column by column.

In [10]:
Z_te = pca.transform(X_te)                    # scores: (transactions x k)
X_te_hat = pca.inverse_transform(Z_te)        # standardized reconstruction: (transactions x 29)
original_eur = escalador.inverse_transform(X_te)          # back to original units
reconstr_eur = escalador.inverse_transform(X_te_hat)
print("Shape of the scores:", Z_te.shape, "  Shape of the reconstruction:", X_te_hat.shape)

i_normal = np.where(y_te == 0)[0][0]
i_fraude = np.where(y_te == 1)[0][0]
mirar = ["V2", "V5", "V7", "V12", "V14", "V17", "V20", "Amount"]
idx = [COLS.index(c) for c in mirar]
comparar = pd.DataFrame({"normal: original": original_eur[i_normal, idx], "normal: reconstructed": reconstr_eur[i_normal, idx],
                         "fraud: original": original_eur[i_fraude, idx], "fraud: reconstructed": reconstr_eur[i_fraude, idx]},
                        index=mirar)
comparar.round(2)

Shape of the scores: (93987, 27)   Shape of the reconstruction: (93987, 29)


,normal: original,normal: reconstructed,fraud: original,fraud: reconstructed
V2,-1.60,-1.62,1.83,-0.59
V5,-1.52,-1.51,-2.74,3.07
V7,-0.81,-0.79,-5.02,5.70
V12,-0.42,-0.40,-9.42,-0.68
V14,-0.15,-0.13,-12.88,-5.11
V17,0.46,0.48,-12.72,-4.35
V20,-0.03,-0.03,2.17,2.12
Amount,175.70,173.82,512.25,488.13


The normal transaction is reconstructed almost identically in every column, including the amount in euros. The fraud
is not: in `V5`, `V7`, `V12`, `V14` and `V17` the reconstruction is far from the original, because with the $k$
directions of normal transactions there is no way to build that combination. That mismatch, summed over the 29
columns, is the suspicion score.

## 6. The fraud detector

### 6.1 The score

A three-line function: the reconstruction error of each transaction. We compute it on training and on
testing with the same PCA (fitted on training).

In [11]:
def error_reconstruccion(modelo, X):
    '''Sum of squared differences between each row and its reconstruction.'''
    return ((X - modelo.inverse_transform(modelo.transform(X))) ** 2).sum(axis=1)

e_tr = error_reconstruccion(pca, X_tr)
e_te = error_reconstruccion(pca, X_te)
print("Error in testing, normal: median", round(np.median(e_te[y_te == 0]), 3), "| 99th percentile", round(np.percentile(e_te[y_te == 0], 99), 2))
print("Error in testing, frauds: median", round(np.median(e_te[y_te == 1]), 3))

fig = go.Figure()
for clase, nombre, color in [(0, "normal", "#64748b"), (1, "fraud", "#dc2626")]:
    fig.add_histogram(x=np.log10(e_te[y_te == clase]), name=nombre, histnorm="probability density", opacity=0.6,
                      marker_color=color, nbinsx=80)
fig.update_layout(barmode="overlay", title="Reconstruction error in testing (log10 scale)",
                  xaxis_title="log10(reconstruction error)", yaxis_title="density", height=380)
fig.show()

Error in testing, normal: median 0.014 | 99th percentile 1.4
Error in testing, frauds: median 201.38


The two distributions are clearly separated: the median error of the frauds is dozens of times
that of the normal ones. But look at the right tail of the normal ones: there are legitimate transactions with errors
as large as a fraud's. An anomaly detector finds what is **unusual**, and not everything unusual is
fraud. With 94 thousand normal transactions in testing, even if only a tiny fraction is unusual, that means hundreds of false alerts.

### 6.2 The threshold without labels

To raise alerts we have to set a threshold. Without labels we cannot search for the threshold that maximizes recall or
F1; what we do know is **how many alerts we can review**. A reasonable rule: alert on the 0.2% most
suspicious, a rate similar to the fraud rate we expect. The threshold is then the **99.8th percentile of the
error in training**, which is computed without looking at any label.

In [12]:
umbral = np.percentile(e_tr, 99.8)
alerta = e_te > umbral
tn, fp, fn, tp = confusion_matrix(y_te, alerta).ravel()
print("Threshold (99.8th percentile of the training error):", round(umbral, 2))
print(f"Alerts in testing: {alerta.sum()}  |  frauds caught (TP): {tp}  |  false alarms (FP): {fp}  |  frauds missed (FN): {fn}")
print(f"Recall: {tp / (tp + fn):.3f}   Precision: {tp / (tp + fp):.3f}")

Threshold (99.8th percentile of the training error): 28.22
Alerts in testing: 190  |  frauds caught (TP): 130  |  false alarms (FP): 60  |  frauds missed (FN): 32
Recall: 0.802   Precision: 0.684


With that threshold the detector generates 190 alerts in testing, of which 130 are fraud: it catches 80% of
fraud (recall 0.80) with 68% of alerts correct (precision 0.68). No number in this cell used a
label to decide; the labels only appear to count the hits.

### 6.3 Evaluating when the labels arrive

Now we do look at the testing labels, only to evaluate. Three measures: the PR-AUC, the AUC of the ROC curve
and the precision among the $k$ most suspicious alerts for several review queue sizes.

In [13]:
def precision_en_k(y, puntaje, ks=(50, 100, 200, 500, 1000)):
    orden = np.argsort(-puntaje)
    return pd.DataFrame({"reviewed (k)": ks,
                         "frauds found": [int(y[orden][:k].sum()) for k in ks],
                         "precision@k": [y[orden][:k].mean() for k in ks],
                         "recall@k": [y[orden][:k].sum() / y.sum() for k in ks]})

print("PR-AUC:", round(average_precision_score(y_te, e_te), 3), " (random would be", round(y_te.mean(), 4), ")")
print("ROC AUC:", round(roc_auc_score(y_te, e_te), 3))
precision_en_k(y_te, e_te).round(3)

PR-AUC: 0.697  (random would be 0.0017 )
ROC AUC: 0.943


,reviewed (k),frauds found,precision@k,recall@k
0,50,41,0.820,0.253
1,100,82,0.820,0.506
2,200,130,0.650,0.802
3,500,130,0.260,0.802
4,1000,131,0.131,0.809


Without using a single label to learn, the detector places **82 frauds among the 100 most
suspicious transactions** in testing, and with 200 reviews it finds 80% of all fraud. The PR-AUC of 0.70 has to be
compared with random, 0.0017: it is about 400 times better than reviewing transactions at random.

Also look at what happens from 200 to 1,000 reviews: the frauds found barely increase (from 130 to 131).
The missing 20% of fraud is not unusual for PCA: these are frauds that look like normal transactions, and
no threshold will find them with this score.

This is the same result reached by the classic example in this course (27 components, reconstruction
error), now measured in testing and with a threshold that does not look at the labels.

## 7. Why does it work? And a warning

The error is the sum of the squared scores on the discarded components (section 3). With $k = 27$
we discard only two: CP28 and CP29. Let us see how much each one contributes to separating frauds from normal transactions.

In [14]:
Z_full_te = pca_full.transform(X_te)
for j in [27, 28]:
    print(f"CP{j + 1}: PR-AUC using only its squared score = {average_precision_score(y_te, Z_full_te[:, j] ** 2):.3f}")
print("\nLoadings of CP28:", top_cargas("CP28").to_dict())

grupo = ["V17", "V12", "V14", "V10", "V16", "V7"]
print("\nCorrelations among these variables, in ALL transactions:")
print(datos[grupo].corr().round(2).iloc[:3].to_string())
print("\nThe same correlations, only in NORMAL transactions:")
print(datos.loc[datos["Class"] == 0, grupo].corr().round(2).iloc[:3].to_string())

CP28: PR-AUC using only its squared score = 0.711


CP29: PR-AUC using only its squared score = 0.012

Loadings of CP28: {'V17': 0.41, 'V12': 0.36, 'V7': 0.35, 'V14': 0.33, 'V10': 0.31, 'V16': 0.29}

Correlations among these variables, in ALL transactions:
     V17  V12  V14  V10  V16   V7
V17  1.0  0.0  0.0  0.0  0.0  0.0
V12  0.0  1.0  0.0  0.0  0.0  0.0
V14  0.0  0.0  1.0  0.0 -0.0  0.0

The same correlations, only in NORMAL transactions:


      V17   V12   V14   V10   V16    V7
V17  1.00 -0.17 -0.16 -0.14 -0.15 -0.14
V12 -0.17  1.00 -0.12 -0.10 -0.09 -0.09
V14 -0.16 -0.12  1.00 -0.10 -0.09 -0.08


Almost all the work is done by **a single component, CP28**: on its own it has a PR-AUC of 0.71, while
CP29 (the one for the amount that does not fit) contributes little. And CP28 loads on `V17`, `V12`, `V14`, `V10`, `V16`, which
are precisely the variables where frauds take extreme values (all very negative at once).

The two correlation tables explain why PCA found that direction. In the full dataset the
`V` columns have **exactly zero** correlation with each other: they are principal components, computed by the issuer
with all transactions, **frauds included**. But if we look only at the normal ones, negative
correlations appear (around minus 0.1 to minus 0.17). For the total to be zero, the frauds, which
all move together in that direction, have to offset them. Result: in normal transactions
the combination `V17 + V12 + V14 + ...` varies **less** than it should, PCA sends it to the bottom of the
list, and by discarding it turns precisely that direction into the detector.

It is a twofold lesson. The good one: PCA finds exactly the directions where normal data barely move, and the
fraud that moves along them becomes visible. The warning: here part of that direction comes from how the dataset was
**anonymized**. With real, non-anonymized data there would not be such a convenient component, and the
result of 0.82 would be more modest.

### 7.1 The choice of $k$ matters

Since only two components do the work, the number we keep is delicate. Looking with the testing
labels (something we could not do in practice, so this is only a diagnostic):

In [15]:
filas = []
for k_ in [5, 10, 15, 20, 25, 26, 27, 28]:
    s = (Z_full_te[:, k_:] ** 2).sum(axis=1)          # error with k_ components = discarded scores squared
    filas.append({"k": k_, "variance kept": var_exp[:k_].sum(), "PR-AUC": average_precision_score(y_te, s),
                  "precision@100": y_te[np.argsort(-s)][:100].mean()})
pd.DataFrame(filas).round(3)

,k,variance kept,PR-AUC,precision@100
0,5,0.210,0.281,0.42
1,10,0.384,0.315,0.44
2,15,0.557,0.335,0.48
3,20,0.729,0.358,0.51
4,25,0.899,0.506,0.62
5,26,0.933,0.575,0.68
6,27,0.966,0.697,0.82
7,28,0.999,0.012,0.01


With $k = 28$ (discarding only CP29) the detector falls to almost nothing, because CP28 is kept and the
fraud is reconstructed well. Between 5 and 26 components the PR-AUC hovers around 0.3 to 0.6. The 95% rule lands on
$k = 27$, the best value, partly by luck. The practical lesson: with anomaly detectors you have to
look at **which components are discarded**, not only how much variance is kept.

## 8. The acid test: learning from the past, detecting in the future

So far we split training and testing at random. But a real detector is trained on yesterday's
transactions and used on tomorrow's. The dataset covers two days and includes `Time`, so we can run the honest
test: **the first 70% of the time is training and the last 30% is testing**.

In [16]:
orden_t = datos.sort_values("Time").reset_index(drop=True)
corte = int(len(orden_t) * 0.70)
tr_t, te_t = orden_t.iloc[:corte], orden_t.iloc[corte:]
yt_tr, yt_te = tr_t["Class"].values, te_t["Class"].values
esc_t = StandardScaler().fit(tr_t[COLS])
Xt_tr, Xt_te = esc_t.transform(tr_t[COLS]), esc_t.transform(te_t[COLS])
print("training:", len(tr_t), "transactions,", yt_tr.sum(), "frauds  |  testing:", len(te_t), "transactions,", yt_te.sum(), "frauds")

pca_t = PCA(n_components=0.95).fit(Xt_tr)
e_t = error_reconstruccion(pca_t, Xt_te)
print("\nPCA with the 95% rule (k =", pca_t.n_components_, ") fitted on ALL of training:")
print("   PR-AUC:", round(average_precision_score(yt_te, e_t), 3), "  precision@100:", yt_te[np.argsort(-e_t)][:100].mean())

training: 199364 transactions, 384 frauds  |  testing: 85443 transactions, 108 frauds

PCA with the 95% rule (k = 27 ) fitted on ALL of training:
   PR-AUC: 0.002   precision@100: 0.0


The detector **collapses**: PR-AUC of 0.002, the same as random, and not a single fraud among the 100 alerts. Same
method, same data, only the way we split time changed. What happened? Let us look for the fraud direction (the
combination of `V17`, `V12`, `V14`...) among the components of this new PCA.

In [17]:
# Diagnostic (uses the training labels only to understand what happened): for each component,
# how much more the frauds vary than the normal ones. The one that separates them most is "the fraud direction".
def direccion_fraude(modelo, X, y):
    Z = modelo.transform(X)
    razon = Z[y == 1].var(axis=0) / Z[y == 0].var(axis=0)
    j = int(np.argmax(razon))
    s = pd.Series(modelo.components_[j], index=COLS)
    return j, razon[j], s.reindex(s.abs().sort_values(ascending=False).index)[:5].round(2).to_dict()

for nombre, ajuste in [("PCA on ALL of training", Xt_tr), ("PCA on normal only", Xt_tr[yt_tr == 0])]:
    m = PCA().fit(ajuste)
    j, razon, top = direccion_fraude(m, Xt_tr, yt_tr)
    print(f"{nombre}: the fraud direction is component {j + 1} of 29 "
          f"(explained variance {m.explained_variance_ratio_[j]:.4f}); frauds vary {razon:.0f} times more than normal ones")
    print("   loadings:", top)
print("Frauds in training:", yt_tr.sum(), "of", len(yt_tr), f"({100 * yt_tr.mean():.2f}%)")

PCA on ALL of training: the fraud direction is component 2 of 29 (explained variance 0.0463); frauds vary 516 times more than normal ones
   loadings: {'V17': 0.39, 'V7': 0.39, 'V14': 0.36, 'V3': 0.34, 'V16': 0.26}
PCA on normal only: the fraud direction is component 28 of 29 (explained variance 0.0073); frauds vary 1039 times more than normal ones
   loadings: {'V17': 0.51, 'V12': 0.38, 'V14': 0.34, 'V10': 0.3, 'V7': 0.28}
Frauds in training: 384 of 199364 (0.19%)


There it is. In this training set the frauds of the first day are very extreme in one direction (`V17`, `V7`, `V14`,
`V3`, `V16`): their score varies more than 500 times as much as that of the normal ones. Although they are only 0.19% of
the rows, that is enough for PCA to see "a lot of variation" in that direction and make it
**component 2**, one of the ones we keep. PCA learned fraud as part of what is normal and, by
reconstructing it well, stopped seeing it. If the same PCA is fitted only on normal transactions, that direction drops to
position 28 of 29, with an explained variance of 0.7%: exactly where we need it.

In the random split this did not happen, by luck: there the training frauds were less extreme as a whole and
the direction ended up in position 28, among the discarded ones.

This is called **contamination** of the training set, and it is the Achilles' heel of any anomaly
detector. The practical solution is to train on transactions we believe are normal: for example, those from more
than 60 days ago that never had a claim. That uses the label in a very weak way (only to exclude
cases), and in the literature it is called *novelty detection* or a semi-supervised approach.

In [18]:
pca_limpio = PCA(n_components=0.95).fit(Xt_tr[yt_tr == 0])     # only normal training transactions
e_limpio = error_reconstruccion(pca_limpio, Xt_te)
print("PCA fitted on NORMAL transactions only (k =", pca_limpio.n_components_, "):")
print("   PR-AUC:", round(average_precision_score(yt_te, e_limpio), 3), "  ROC AUC:", round(roc_auc_score(yt_te, e_limpio), 3))
precision_en_k(yt_te, e_limpio).round(3)

PCA fitted on NORMAL transactions only (k = 26 ):
   PR-AUC: 0.589   ROC AUC: 0.888


,reviewed (k),frauds found,precision@k,recall@k
0,50,44,0.880,0.407
1,100,76,0.760,0.704
2,200,83,0.415,0.769
3,500,84,0.168,0.778
4,1000,84,0.084,0.778


With a clean training set the detector works again in the future: PR-AUC of 0.59 and 76 frauds among the
100 alerts. It is somewhat less than with the random split (0.70), as expected when the future does not
fully resemble the past.

## 9. What if we had labels? The comparison with supervised models

The mandatory question: how much do we lose by not using labels? We train the two supervised models from
previous notebooks with the **same columns and splits**: a logistic regression and a regularized gradient
boosting (trees of depth 3, learning rate 0.05, as we learned in
notebook 6). We add a third, realistic case: a logistic regression that only knows **20 labeled frauds**
(the first ones that filed claims), plus all the normal transactions.

In [19]:
def supervisados(Xa, ya, Xb, semilla=0):
    out = {}
    out["logistic (all labels)"] = LogisticRegression(max_iter=2000).fit(Xa, ya).decision_function(Xb)
    out["gradient boosting (all labels)"] = HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=300, l2_regularization=1.0, min_samples_leaf=50,
        random_state=semilla).fit(Xa, ya).predict_proba(Xb)[:, 1]
    rng = np.random.default_rng(semilla)
    pocos = np.r_[rng.choice(np.where(ya == 1)[0], 20, replace=False), np.where(ya == 0)[0]]
    out["logistic with only 20 frauds"] = LogisticRegression(max_iter=2000).fit(Xa[pocos], ya[pocos]).decision_function(Xb)
    return out

def resumen(y, puntajes):
    return pd.DataFrame([{"model": m, "PR-AUC": average_precision_score(y, s), "ROC AUC": roc_auc_score(y, s),
                          "precision@100": y[np.argsort(-s)][:100].mean(),
                          "recall@500": y[np.argsort(-s)][:500].sum() / y.sum()} for m, s in puntajes.items()]).set_index("model")

azar = {"PCA without labels (all training)": e_te, "PCA normal only": error_reconstruccion(PCA(n_components=0.95).fit(X_tr[y_tr == 0]), X_te)}
azar.update(supervisados(X_tr, y_tr, X_te))
tiempo = {"PCA without labels (all training)": e_t, "PCA normal only": e_limpio}
tiempo.update(supervisados(Xt_tr, yt_tr, Xt_te))

tabla = pd.concat({"random split": resumen(y_te, azar), "time split": resumen(yt_te, tiempo)}, axis=1)
tabla.round(3)

random split                                  time split                                 
                                        PR-AUC ROC AUC precision@100 recall@500     PR-AUC ROC AUC precision@100 recall@500
model                                                                                                                      
PCA without labels (all training)        0.697   0.943          0.82      0.802      0.002   0.595          0.00      0.000
PCA normal only                          0.579   0.963          0.65      0.802      0.589   0.888          0.76      0.778
logistic (all labels)                    0.770   0.978          0.85      0.883      0.705   0.973          0.72      0.843
gradient boosting (all labels)           0.863   0.981          0.98      0.877      0.759   0.979          0.81      0.815
logistic with only 20 frauds             0.688   0.952          0.81      0.833      0.520   0.950          0.52      0.769

In [20]:
fig = go.Figure()
for nombre, s in tiempo.items():
    p, r, _ = precision_recall_curve(yt_te, s)
    fig.add_scatter(x=r, y=p, mode="lines", name=f"{nombre} ({average_precision_score(yt_te, s):.2f})")
fig.update_layout(title="Precision-recall curves in the future (time split); PR-AUC in parentheses",
                  xaxis_title="recall", yaxis_title="precision", height=450)
fig.show()

How to read the table:

- **With all the labels, supervised wins.** The regularized gradient boosting reaches a PR-AUC of
  0.86 in the random split (98 frauds among the first 100 alerts) and 0.76 in the future; the logistic regression, 0.77 and 0.71.
  The best PCA stays at 0.70 in the random split and 0.59 in the future. If you have abundant and reliable labels, use them.
- **In the future the gap narrows.** In precision@100, what the fraud team sees, the clean PCA
  (0.76) beats the logistic regression with all labels (0.72) and comes close to boosting (0.81).
- **With few labels, it depends.** The logistic regression that knows only 20 frauds ties with PCA in the random
  split (0.69 versus 0.70) but drops to 0.52 in the future, below the clean PCA (0.59). Few labels
  generalize poorly to fraud that changes; PCA does not depend on which frauds it happened to get.
- **The contaminated PCA is the only one that collapses.** Its problem is not the algorithm but what we told it
  was normal. (In the random split, the PCA on normal only stays at 0.58, below the contaminated one: the 95% rule
  picks 26 components there instead of 27, and as we saw in section 7.1, one component more or
  less changes the result a lot.)

## 10. Key takeaways

1. PCA finds the directions in which the data vary; its pieces in scikit-learn are
   `explained_variance_ratio_` (how much each direction explains), `components_` (the loadings, which variables
   weigh in each one), `transform` (the scores) and `inverse_transform` (the reconstruction), and
   `escalador.inverse_transform` takes the reconstruction back to the original units.
2. The **reconstruction error** is the sum of the squared scores on the discarded components: it measures
   how much a transaction moves in the directions where normal ones barely move. It does not look for what is
   large, it looks for what **breaks the relationships** of what is normal.
3. Without labels, the threshold comes from the **review capacity** (a percentile of the training error), and
   evaluation waits for the labels to arrive.
4. The detector depends critically on **what is "normal" to it**. Trained with frauds inside, it can
   learn fraud as one more pattern and stop seeing it. Training on transactions verified as
   normal fixes it.
5. With abundant labels, supervised wins; with few labels or when the future changes, the unsupervised
   detector is competitive, and in practice fraud teams use both at the same time.
6. Part of the good result on this dataset comes from how the data were anonymized. You always have to ask
   **why** a detector works, not only **how much**.

In notebook 10 we replace PCA with a neural network that does the same thing, an **autoencoder**, and
check whether nonlinearity buys anything.

## Exercises

1. **The clock as a variable.** In section 4 we left out `Time` because it only grows. Put it to the test:
   repeat the detector from section 6 (random split) and the one from section 8 (time split, PCA
   on normal only) adding `Time` to the columns. Before running, predict what will happen in the time
   split, where all of testing has hours that training never saw. Then explain the result using the
   error formula from section 3.
2. **The threshold as a decision.** With the clean PCA from the time split, compute recall, precision and
   number of alerts in testing for thresholds at the 99th, 99.5th, 99.8th and 99.9th percentiles of the training error.
   If each review costs 5 euros and each undetected fraud costs its amount (`Amount`), which of the four
   thresholds minimizes the total cost?
3. **How many labels are worth as much as PCA?** In the time split, train the logistic regression with
   5, 10, 20, 50, 100 and 200 randomly chosen frauds (plus all the normal transactions) and plot its PR-AUC in testing
   against the number of frauds, with a horizontal line at the PR-AUC of the clean PCA. From how many
   labels on is supervised worth it? Repeat with three seeds to see how much it moves.

## Solutions

In [21]:
# Exercise 1: adding Time
COLS_T = ["Time"] + COLS
for nombre, (a, b) in {"random": (tr, te), "time": (tr_t, te_t)}.items():
    esc = StandardScaler().fit(a[COLS_T])
    A, B = esc.transform(a[COLS_T]), esc.transform(b[COLS_T])
    if nombre == "time":
        A = A[a["Class"].values == 0]                       # normal only, as in section 8
    m = PCA(n_components=0.95).fit(A)
    s = error_reconstruccion(m, B)
    print(f"{nombre:10s} with Time: k = {m.n_components_}, PR-AUC = {average_precision_score(b['Class'], s):.3f}, "
          f"precision@100 = {b['Class'].values[np.argsort(-s)][:100].mean():.2f}")
    print(f"           standardized Time in testing: minimum {B[:, 0].min():.2f}, maximum {B[:, 0].max():.2f}")

random     with Time: k = 27, PR-AUC = 0.695, precision@100 = 0.82
           standardized Time in testing: minimum -2.00, maximum 1.64


time       with Time: k = 26, PR-AUC = 0.586, precision@100 = 0.76
           standardized Time in testing: minimum 1.81, maximum 2.97


The natural prediction is that in the time split the detector breaks down: all of testing has standardized
`Time` values between 1.8 and 3.0, larger than almost any seen in training. But the
result barely changes (PR-AUC 0.586 versus 0.589). The formula from section 3 explains it: the error only
adds up the scores of the **discarded** components. `Time` is almost uncorrelated with the other
columns, so it forms its own component with variance close to 1, which ends up among the **kept** ones. The
testing transactions move far away in that direction, but PCA reconstructs them well along it, and the
error does not register it.

That does not mean including the clock is harmless. Other anomaly detectors that measure distance in
**all** directions (the Mahalanobis distance, an Isolation Forest) would flag any
late transaction as unusual. And if the clock were correlated with other variables, it would fall into the
discarded components. Variables that only grow (dates, counters, accumulated tenure) are best
transformed (hour of day, day of week) or left out.

In [22]:
# Exercise 2: the threshold as a decision with costs
e_tr_limpio = error_reconstruccion(pca_limpio, Xt_tr[yt_tr == 0])
montos = te_t["Amount"].values
filas = []
for q in [99, 99.5, 99.8, 99.9]:
    u = np.percentile(e_tr_limpio, q)
    a = e_limpio > u
    tp, fp = (a & (yt_te == 1)).sum(), (a & (yt_te == 0)).sum()
    fn_monto = montos[np.logical_not(a) & (yt_te == 1)].sum()
    filas.append({"percentile": q, "alerts": int(a.sum()), "recall": tp / yt_te.sum(), "precision": tp / max(a.sum(), 1),
                  "review cost": 5 * a.sum(), "undetected fraud (EUR)": fn_monto, "total cost": 5 * a.sum() + fn_monto})
pd.DataFrame(filas).round(3)

,percentile,alerts,recall,precision,review cost,undetected fraud (EUR),total cost
0,99.0,2758,0.806,0.032,13790,2534.71,16324.71
1,99.5,1392,0.778,0.060,6960,2538.70,9498.70
2,99.8,547,0.778,0.154,2735,2538.70,5273.70
3,99.9,293,0.769,0.283,1465,2539.70,4004.70


The result contradicts the intuition that "a fraud costs much more than a review, so it pays to
alert more". From the 99th to the 99.9th percentile the alerts drop from 2,758 to 293, but recall barely moves
(from 0.81 to 0.77) and the amount of undetected fraud is practically the same, about 2,540 euros. The reason is
the one we saw in section 6.3: the frauds PCA does not see are frauds that look normal, and lowering the
threshold does not reach them; it only adds false alarms. That is why the **strictest** threshold of the four is the
cheapest. Before opening up the threshold you have to look at the recall curve: if it is flat, each extra alert is almost
purely wasted review.

In [23]:
# Exercise 3: how many labels are worth as much as PCA
ap_pca = average_precision_score(yt_te, e_limpio)
filas = []
for semilla in [0, 1, 2]:
    rng = np.random.default_rng(semilla)
    for n1 in [5, 10, 20, 50, 100, 200]:
        sel = np.r_[rng.choice(np.where(yt_tr == 1)[0], n1, replace=False), np.where(yt_tr == 0)[0]]
        s = LogisticRegression(max_iter=2000).fit(Xt_tr[sel], yt_tr[sel]).decision_function(Xt_te)
        filas.append({"seed": semilla, "labeled frauds": n1, "PR-AUC": average_precision_score(yt_te, s)})
curva_et = pd.DataFrame(filas)
fig = px.line(curva_et, x="labeled frauds", y="PR-AUC", color=curva_et["seed"].astype(str), markers=True, log_x=True,
              title="Logistic regression by how many frauds it knows, versus the clean PCA (dotted line)")
fig.add_hline(y=ap_pca, line_dash="dot", annotation_text=f"PCA normal only: {ap_pca:.2f}")
fig.update_layout(height=400, legend_title="seed")
fig.show()
curva_et.pivot(index="labeled frauds", columns="seed", values="PR-AUC").round(3)

seed,0,1,2
labeled frauds,,,
5,0.479,0.370,0.558
10,0.510,0.634,0.451
20,0.490,0.522,0.540
50,0.543,0.560,0.549
100,0.560,0.577,0.563
200,0.617,0.646,0.602


With 5 to 100 labeled frauds the logistic regression stays almost always below the clean PCA (0.59), and with very
few labels its result depends a lot on **which** frauds it happened to get: with 5 it ranges from 0.37 to 0.56 depending on the
seed. Only with about 200 frauds does it beat PCA in all three seeds, and with the 384 in training it reaches 0.71.
The exact crossover moves with the seed; what does not change is the shape of the curve. In a new system, or against a type of fraud that has no claims yet, the
unsupervised detector is the day-one tool, and the supervised model takes over once the
labels accumulate.